# 🤖 03 - Huấn luyện Model (Model Training)
**EduTalk HUIT — Hệ thống Tư vấn Ngành học**

Notebook này thực hiện:
- Train **XGBoost** (model chính — phân loại đa lớp)
- Train **Cosine Similarity** (tìm ngành gần nhất theo tổ hợp điểm)
- Hyperparameter tuning (GridSearchCV)
- Export model ra file `.pkl` để deploy

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

print('✅ Libraries loaded')

## 1. Load dữ liệu đã xử lý

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val   = pd.read_csv('../data/processed/X_val.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val   = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()
le      = joblib.load('../models/label_encoder.pkl')

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Số ngành: {y_train.nunique()}')

## 2. Baseline — So sánh nhanh các model

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42,
                                         eval_metric='mlogloss', verbosity=0),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores
    print(f'{name:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

## 3. Train XGBoost với Hyperparameter Tuning

In [ ]:
# Tìm siêu tham số tốt nhất
param_grid = {
    'n_estimators':   [100, 200, 300],
    'max_depth':      [3, 5, 7],
    'learning_rate':  [0.05, 0.1, 0.2],
    'subsample':      [0.8, 1.0],
}

xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)
grid = GridSearchCV(xgb, param_grid, cv=5, scoring='accuracy',
                    n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f'\n✅ Best params: {grid.best_params_}')
print(f'✅ Best CV accuracy: {grid.best_score_:.4f}')

## 4. Train model tốt nhất và đánh giá trên Validation set

In [ ]:
best_xgb = grid.best_estimator_
best_xgb.fit(X_train, y_train)

y_pred_val = best_xgb.predict(X_val)
val_acc = accuracy_score(y_val, y_pred_val)
print(f'Validation Accuracy: {val_acc:.4f}\n')
print(classification_report(
    y_val, y_pred_val,
    target_names=le.classes_
))

## 5. Cosine Similarity — Gợi ý ngành theo vector điểm

In [ ]:
# Tính vector trung bình điểm của mỗi ngành trong tập train
X_train_copy = X_train.copy()
X_train_copy['label'] = y_train.values

major_profiles = X_train_copy.groupby('label').mean()

def predict_cosine(score_vector: list, top_k: int = 3):
    """Trả về top-k ngành gần nhất theo Cosine Similarity."""
    vec = np.array(score_vector).reshape(1, -1)
    sims = cosine_similarity(vec, major_profiles.values)[0]
    top_idx = sims.argsort()[::-1][:top_k]
    return [(le.classes_[i], round(sims[i], 4)) for i in top_idx]

# Ví dụ test
sample = X_val.iloc[0].tolist()
print('Top-3 ngành gợi ý (Cosine Similarity):')
for major, score in predict_cosine(sample):
    print(f'  {major}: {score}')

## 6. Lưu model

In [ ]:
import joblib
joblib.dump(best_xgb,       '../models/xgboost_model.pkl')
joblib.dump(major_profiles, '../models/cosine_profiles.pkl')

print('✅ Đã lưu:')
print('   models/xgboost_model.pkl')
print('   models/cosine_profiles.pkl')
print('→ Chạy tiếp: 04_evaluation.ipynb')